# Flow Matching on $\mathbb{P}_3$
In this notebook, we perform flow matching on Event-Velocity space $\mathbb{P}_3$.

In [ ]:
import torch
import numpy as np
from math import sqrt
from lieflow.groups import P3
from lieflow.models import (
    get_model_FM,
    LogarithmicDistance
)
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.cm import ScalarMappable
from tqdm.notebook import tqdm
from functools import partial
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern"],
    "font.size": 10.0,
    "text.latex.preamble": r"\usepackage{lmodern} \usepackage{amssymb} \usepackage{amsmath}"
})
%matplotlib widget

In [ ]:
p3 = P3()

In [ ]:
generate_videos = True

# Distributions

In [ ]:
def random_time_like(N, ε_stab=1e-4):
    vec = torch.randn(N, 3)
    vec[..., 0] = (1 + torch.randn(N)**2).sqrt().clamp_min(1 + ε_stab) * (vec[..., 1]**2 + vec[..., 2]**2).sqrt()
    # restrict such that |angle| < pi
    return vec * ((1 - 1000. * ε_stab) * torch.pi / inner_product(vec, vec, keepdim=True).sqrt()).clamp_max(1.)

def random_space_like(N):
    vec = torch.randn(N, 3)
    vec[..., 0] = torch.clamp_min(1/(1 + torch.randn(N)**2).sqrt(), 0.1) * (vec[..., 1]**2 + vec[..., 2]**2).sqrt()
    return vec

def random_light_like(N):
    vec = torch.randn(N, 3)
    vec[..., 0] = (vec[..., 1]**2 + vec[..., 2]**2).sqrt()
    return vec

def inner_product(vec1, vec2, keepdim=False):
    return (vec1 @ p3.η * vec2).sum(-1, keepdim=keepdim)

def random_unit_time_like(N):
    vec = torch.randn(N, 3)
    vec[..., 0] = (1 + vec[..., 1]**2 + vec[..., 2]**2).sqrt()
    return vec

def generate_line(N, g, w=1., ε=0.05):
    x = torch.zeros(N, 3)
    x[:, 1] = 2. * w * (torch.rand(N) - 0.5)
    x += ε * torch.randn(N, 3)
    n = torch.zeros(N, 3)
    n[:, 0] = 1.
    n += ε * torch.randn(N, 3)
    n = n / (n @ p3.η * n).sum(-1, keepdim=True).sqrt()
    p = p3.pack_position_orientation(x, n)
    return p3.act(g, p)

def generate_normals(N, g):
    xs = torch.randn(N, 3)
    ns = torch.randn(N, 3)
    ns[..., 0] = (1 + ns[..., 1]**2 + ns[..., 2]**2).sqrt()
    return p3.act(g, p3.pack_position_orientation(xs, ns))

x_ref = torch.zeros(3)
n_ref_t = torch.tensor([1., 0., 0.])
n_ref_x = torch.tensor([0., 1., 0.])
n_ref_y = torch.tensor([0., 0., 1.])
p_ref = p3.pack_position_orientation(x_ref, n_ref_t)
p_ref_x = p3.pack_position_orientation(x_ref, n_ref_x)
p_ref_y = p3.pack_position_orientation(x_ref, n_ref_y)

def g_to_frame(g):
    return p3.act(g, p_ref), p3.act(g, p_ref_x), p3.act(g, p_ref_y)

def lift_to_SEplus12(p):
    A = p3.get_generator(p_ref, p)
    random_rotation = p3.seplus12.exp(2 * torch.pi * torch.rand(*p.shape[:-2], 1, 1) * p3.seplus12.lie_algebra_basis[-3])
    return p3.seplus12.L(p3.seplus12.exp(A), random_rotation)
    
def project_to_M3(g):
    return p3.act(g, p_ref)

def plot_p(ax, p, **kwargs):
    x, n = p3.get_position_orientation(p)
    return ax.quiver(x[..., 0], x[..., 1], x[..., 2], n[..., 0], n[..., 1], n[..., 2], **kwargs)

def plot_g(ax, g, **kwargs):
    p_t, p_x, p_y = g_to_frame(g)
    plot_1 = plot_p(ax, p_t, color="red", **kwargs)
    plot_2 = plot_p(ax, p_x, color="green", **kwargs)
    plot_3 = plot_p(ax, p_y, color="blue", **kwargs)
    return plot_1, plot_2, plot_3

In [ ]:
tests = ("line_to_line", "normals_to_line")

In [ ]:
def data_generator(test, ε=0.04):
    match test:
        case "line_to_line":
            g_0 = p3.seplus12.exp((torch.tensor([0., 0., -2., 0., 0., 0.])[..., None, None] * p3.seplus12.lie_algebra_basis).sum(0))
            g_1 = p3.seplus12.exp((torch.tensor([0., 0., 2., 0., 0.2, 0.2])[..., None, None] * p3.seplus12.lie_algebra_basis).sum(0))
            generate_p_0 = partial(generate_line, g=g_0, ε=ε)
            generate_p_1 = partial(generate_line, g=g_1, ε=ε)
        case "normals_to_line":
            g_0 = p3.seplus12.exp((torch.tensor([0., -1., 0., 0., 0., 0.])[..., None, None] * p3.seplus12.lie_algebra_basis).sum(0))
            g_1 = p3.seplus12.exp((torch.tensor([0., 1., 0., 0., 0., 0.])[..., None, None] * p3.seplus12.lie_algebra_basis).sum(0))
            generate_p_0 = partial(generate_normals, g=g_0)
            generate_p_1 = partial(generate_line, g=g_1, ε=ε)
    return generate_p_0, generate_p_1

In [ ]:
generate_p_0, generate_p_1 = data_generator(tests[0])

def generate_g_0(N): return lift_to_SEplus12(generate_p_0(N))
def generate_g_1(N): return lift_to_SEplus12(generate_p_1(N))

In [ ]:
l_big = 0.4
l_small = 0.2
alpha_big = 0.8
alpha_small = 0.3

p_0s = generate_p_0(32)
g_0s = lift_to_SEplus12(p_0s)
p_0_x, p_0_y, p_0_z = g_to_frame(g_0s)
p_1s = generate_p_1(32)
g_1s = lift_to_SEplus12(p_1s)

# g_0s = generate_g_0(32)
# p_0s = project_to_M3(g_0s)
# p_0_x, p_0_y, p_0_z = g_to_frame(g_0s)
# g_1s = generate_g_1(32)
# p_1s = project_to_M3(g_1s)

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection="3d")
plot_p(ax, p_0s, alpha=alpha_big, color="red", length=l_big)
plot_p(ax, p_1s, alpha=alpha_big, color="green", length=l_big)
plot_g(ax, g_0s, alpha=alpha_small, length=l_small)
plot_g(ax, g_1s, alpha=alpha_small, length=l_small)
ax.set_xlim(-3, 3)
ax.set_xlabel("$t$")
ax.set_ylim(-3, 3)
ax.set_ylabel("$x$")
ax.set_zlim(-3, 3)
ax.set_zlabel("$y$");

# Models

## Training

In [ ]:
EPSILON = 0.01
N = 2**14
BATCH_SIZE = 2**10
EPOCHS = len(tests) * [50]
WEIGHT_DECAY = 0.
LEARNING_RATE = 1e-3
H = 64 # Width
L = 3 # Number of layers is L + 2
device = "cpu"

In [ ]:
def train_models(p_0s, p_1s, epochs, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY):
    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(p_0s, p_1s), batch_size=batch_size, shuffle=True
    )

    model_P3 = get_model_FM(p3, H=H, L=L).to(device)
    print("Number of parameters: ", model_P3.parameter_count)
    optimizer_P3 = torch.optim.Adam(model_P3.parameters(), learning_rate, weight_decay=weight_decay)
    loss = LogarithmicDistance(torch.tensor([1., 1., 1., 1., 1., 1.]))

    losses_P3 = np.zeros(epochs)
    for i in tqdm(range(epochs)):
        losses_P3[i] = model_P3.train_network(device, train_loader, optimizer_P3, loss)
    
    g_0s = lift_to_SEplus12(p_0s)
    g_1s = lift_to_SEplus12(p_1s)
    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(g_0s, g_1s), batch_size=batch_size, shuffle=True
    )

    model_SEplus12 = get_model_FM(p3.seplus12, H=H, L=L).to(device)
    print("Number of parameters: ", model_SEplus12.parameter_count)
    optimizer_SEplus12 = torch.optim.Adam(model_SEplus12.parameters(), learning_rate, weight_decay=weight_decay)
    loss = LogarithmicDistance(torch.tensor([1., 1., 1., 1., 1., 1.]))

    losses_SEplus12 = np.zeros(epochs)
    for i in tqdm(range(epochs)):
        losses_SEplus12[i] = model_SEplus12.train_network(device, train_loader, optimizer_SEplus12, loss)

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.plot(losses_P3, label=r"$\mathbb{P}_3$")
    ax.plot(losses_SEplus12, label=r"$\operatorname{SE}^+(1, 2)$")
    ax.set_title("Batch Loss Flow Matching")
    ax.legend()
    ax.set_xscale("log")
    ax.set_yscale("log")

    return model_P3, model_SEplus12

In [ ]:
models_P3 = {}
models_SEplus12 = {}
for i, test in enumerate(tests):
    print(test)
    generate_p_0, generate_p_1 = data_generator(test, ε=EPSILON)
    
    p_0s = generate_p_0(N)
    g_0s = lift_to_SEplus12(p_0s)
    p_1s = generate_p_1(N)
    g_1s = lift_to_SEplus12(p_1s)
    p_0_x, p_0_y, p_0_z = g_to_frame(g_0s)
    p_1_x, p_1_y, p_1_z = g_to_frame(g_1s)

    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')
    plot_p(ax, p_0s[:32], alpha=alpha_big, color="red", length=l_big)
    plot_p(ax, p_1s[:32], alpha=alpha_big, color="green", length=l_big)
    plot_g(ax, g_0s[:32], alpha=alpha_small, length=l_small)
    plot_g(ax, g_1s[:32], alpha=alpha_small, length=l_small)
    ax.set_xlim(-3, 3)
    ax.set_xlabel("$x$")
    ax.set_ylim(-3, 3)
    ax.set_ylabel("$y$")
    ax.set_zlim(-3, 3)
    ax.set_zlabel("$z$")
    models_P3[test], models_SEplus12[test] = train_models(
        p_0s,
        p_1s,
        epochs=EPOCHS[i],
        batch_size=BATCH_SIZE,                                                    learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

## Testing

In [ ]:
def create_animations(tests, models_P3, models_SEplus12, N=2**5):
    p_0s_test = {}
    p_ts_test = {}
    g_0s_test = {}
    g_ts_test = {}
    for test in tests:
        generate_p_0, _ = data_generator(test, ε=EPSILON)
        p_0s_test[test] = generate_p_0(N).to(device)
        p_ts_test[test] = p_0s_test[test].detach().clone()
        g_0s_test[test] = lift_to_SEplus12(p_0s_test[test])
        g_ts_test[test] = g_0s_test[test].detach().clone()
        
        models_P3[test].eval()
        models_SEplus12[test].eval()

    t = 0
    N_steps = 120
    Δt = 1. / N_steps

    for test in tests:
        metadata = {'title': f'Flow Matching P3 {test}', 'artist': 'Matplotlib'}
        writer = FFMpegWriter(fps=30, metadata=metadata)

        fig = plt.figure(figsize=(10, 10))
        ax = fig.add_subplot(111, projection='3d')

        ax.set_xlim(-3, 3)
        ax.set_xlabel("$t$")
        ax.set_ylim(-3, 3)
        ax.set_ylabel("$x$")
        ax.set_zlim(-3, 3)
        ax.set_zlabel("$y$")
        with writer.saving(fig, f"output/flow_matching_P3_{test}.mp4", dpi=150):
            q_0 = ax.quiver([], [], [], [], [], [])
            for frame in tqdm(range(N_steps+1)):
                q_0.remove()
                q_0 = plot_p(ax, p_ts_test[test].to("cpu"), length=l_big)
                writer.grab_frame()
                t = frame * Δt
                with torch.no_grad():
                    p_ts_test[test] = models_P3[test].step(p_ts_test[test], torch.Tensor([t])[..., None, None], Δt)
            
        metadata = {'title': f'Flow Matching SE^+(1, 2) {test}', 'artist': 'Matplotlib'}
        writer = FFMpegWriter(fps=30, metadata=metadata)

        fig = plt.figure(figsize=(10, 10))
        ax = fig.add_subplot(111, projection='3d')

        ax.set_xlim(-3, 3)
        ax.set_xlabel("$t$")
        ax.set_ylim(-3, 3)
        ax.set_ylabel("$x$")
        ax.set_zlim(-3, 3)
        ax.set_zlabel("$y$")
        with writer.saving(fig, f"output/flow_matching_SEplus12_{test}.mp4", dpi=150):
            artists = []
            for frame in tqdm(range(N_steps+1)):
                for a in artists:
                    a.remove()
                artists = plot_g(ax, g_ts_test[test].to("cpu"), alpha=alpha_small, length=l_small)
                writer.grab_frame()
                t = frame * Δt
                with torch.no_grad():
                    g_ts_test[test] = models_SEplus12[test].step(g_ts_test[test], torch.Tensor([t])[..., None, None], Δt)


In [ ]:
if generate_videos:
    create_animations(tests, models_P3, models_SEplus12, N=2**5)